# Walk-Forward Backtest Notebook (EXP: multi-asset + multi-timeframe)

This notebook keeps the v1 pipeline structure and extends it to:
- assets: BTCUSDT, ETHUSDT, BNBUSDT, XRPUSDT, DOGEUSDT
- timeframes: 1d and 4h

`runner.ipynb` remains the immutable v1 baseline.


In [4]:
# ==== Setup (imports, logging, versions) ====
import logging
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("wf_notebook_exp")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Run notebook from project root. cwd={PROJECT_ROOT}")

from src.metrics import compute_metrics_from_returns
from src.experiment_config import (
    periods_per_year_for_timeframe,
    warmup_bars_for_timeframe,
)
from src.artifacts import safe_write_parquet
from src.pipeline_exp import (
    build_timeframe_context,
    run_wf_phase,
    run_reports_phase,
    run_tests_phase,
    run_robustness_phase,
)

np.random.seed(42)
logger.info("PROJECT_ROOT=%s", PROJECT_ROOT)


2026-04-02 14:13:49,182 | INFO | PROJECT_ROOT=c:\Users\prosh\OneDrive\Рабочий стол\git\diploma


In [5]:
# ==== Experiment Config (multi-timeframe) ====
TIMEFRAMES = ["1d", "4h"]
SYMBOL_TO_ACTIVE = {
    "BTCUSDT": "btc",
    "ETHUSDT": "eth",
    "BNBUSDT": "bnb",
    "XRPUSDT": "xrp",
    "DOGEUSDT": "doge",
}

OBJECTIVE_MIN_TRADES = {"1d": 10, "4h": 30}
RC_BLOCK_SIZE = {"1d": 10, "4h": 60}
ROBUST_MIN_BARS = {"1d": 50, "4h": 300}

LAMBDA_LIST = [0.5, 1.0, 2.0]
LAM_STAR = 1.0
ALPHA = 0.05
MAX_SHOW = 20
PV_TYPE = "consistent"

SUBPERIODS = {
    "2025-10_to_2026-02": ("2025-10-01", "2026-03-01"),
    "2022H1": ("2022-01-01", "2022-07-01"),
    "2022-07_to_2022-11": ("2022-07-01", "2022-12-01"),
}

plan_rows = []
for tf in TIMEFRAMES:
    for symbol, active in SYMBOL_TO_ACTIVE.items():
        plan_rows.append({
            "timeframe": tf,
            "symbol": symbol,
            "csv": str(PROJECT_ROOT / "data" / f"{active}-{tf}.csv"),
            "periods_per_year": periods_per_year_for_timeframe(tf),
            "warmup_bars": warmup_bars_for_timeframe(tf),
            "min_trades": OBJECTIVE_MIN_TRADES[tf],
            "rc_block": RC_BLOCK_SIZE[tf],
            "robust_min_bars": ROBUST_MIN_BARS[tf],
        })

plan_df = pd.DataFrame(plan_rows)
display(plan_df)


,timeframe,symbol,csv,periods_per_year,warmup_bars,min_trades,rc_block,robust_min_bars
0,1d,BTCUSDT,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...,365,300,10,10,50
1,1d,ETHUSDT,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...,365,300,10,10,50
2,1d,BNBUSDT,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...,365,300,10,10,50
3,1d,XRPUSDT,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...,365,300,10,10,50
4,1d,DOGEUSDT,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...,365,300,10,10,50
5,4h,BTCUSDT,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...,2190,1800,30,60,300
6,4h,ETHUSDT,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...,2190,1800,30,60,300
7,4h,BNBUSDT,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...,2190,1800,30,60,300
8,4h,XRPUSDT,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...,2190,1800,30,60,300
9,4h,DOGEUSDT,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...,2190,1800,30,60,300


## Run Full Pipeline For Each Timeframe

### Phase 1. WF Runs


In [6]:
contexts = {}
wf_outputs = {}

for tf in TIMEFRAMES:
    logger.info("===== PHASE 1/4 | WF RUNS | %s =====", tf)
    ctx = build_timeframe_context(
        project_root=PROJECT_ROOT,
        timeframe=tf,
        symbol_to_active=SYMBOL_TO_ACTIVE,
        objective_min_trades=OBJECTIVE_MIN_TRADES,
        rc_block_size=RC_BLOCK_SIZE,
        results_root_name="runner_exp",
        logger=logger,
    )
    wf = run_wf_phase(ctx, logger=logger)
    contexts[tf] = ctx
    wf_outputs[tf] = wf
    logger.info(
        "WF DONE %s | folds_best=%s | returns_oos=%s",
        tf,
        wf.folds_best.shape,
        wf.returns_oos.shape,
    )

list(wf_outputs.keys())


2026-04-02 14:14:05,106 | INFO | ===== PHASE 1/4 | WF RUNS | 1d =====
2026-04-02 14:14:05,144 | INFO | BTCUSDT loaded: n=3088 | 2017-09-16 .. 2026-02-28
2026-04-02 14:14:05,178 | INFO | ETHUSDT loaded: n=3088 | 2017-09-16 .. 2026-02-28
2026-04-02 14:14:05,215 | INFO | BNBUSDT loaded: n=3007 | 2017-12-06 .. 2026-02-28
2026-04-02 14:14:05,250 | INFO | XRPUSDT loaded: n=2828 | 2018-06-03 .. 2026-02-28
2026-04-02 14:14:05,278 | INFO | DOGEUSDT loaded: n=2401 | 2019-08-04 .. 2026-02-28
2026-04-02 14:14:05,283 | INFO | 1d | BTCUSDT folds=10
2026-04-02 14:14:05,288 | INFO | 1d | ETHUSDT folds=10
2026-04-02 14:14:05,292 | INFO | 1d | BNBUSDT folds=10
2026-04-02 14:14:05,296 | INFO | 1d | XRPUSDT folds=9
2026-04-02 14:14:05,299 | INFO | 1d | DOGEUSDT folds=7
2026-04-02 14:14:05,300 | INFO | RUN: BTCUSDT | cost=Low | folds=10 | strategies=12
2026-04-02 14:17:11,753 | INFO | RUN: BTCUSDT | cost=Base | folds=10 | strategies=12
2026-04-02 14:20:16,630 | INFO | RUN: BTCUSDT | cost=High | folds=10 | 

['1d', '4h']

In [7]:
# Cell A: Save Phase 1 snapshot (artifacts + reproducibility context)
import json
import shutil
import hashlib
import pickle
import subprocess
import platform
from pathlib import Path
from dataclasses import asdict

import pandas as pd

SAVE_PICKLE_CACHE = True  # set False if you do not want large pickle files

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def git_head_or_none() -> str | None:
    try:
        out = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
        return out
    except Exception:
        return None

timestamp = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M%S")
PHASE1_SNAPSHOT_ROOT = PROJECT_ROOT / "results" / "runner_exp" / "_snapshots" / f"phase1_{timestamp}"
PHASE1_SNAPSHOT_ROOT.mkdir(parents=True, exist_ok=True)

summary_rows = []

for tf in TIMEFRAMES:
    ctx = contexts[tf]
    wf = wf_outputs[tf]
    tf_snap = PHASE1_SNAPSHOT_ROOT / tf
    tf_snap.mkdir(parents=True, exist_ok=True)

    # 1) Copy core Phase 1 artifacts to immutable snapshot
    phase1_files = [
        "folds_best.parquet",
        "returns_oos.parquet",
        "bench_returns_oos.parquet",
        "bench_folds.parquet",
        "train_heatmaps.parquet",
        f"config_notebook_{tf}.json",
    ]
    copied = []
    for name in phase1_files:
        src = ctx.cfg.results_dir / name
        if src.exists():
            shutil.copy2(src, tf_snap / name)
            copied.append(name)

    # 2) Save fold windows snapshot
    fold_rows = []
    for sym, folds in ctx.folds_by_symbol.items():
        for f in folds:
            fold_rows.append({
                "symbol": sym,
                "fold_id": int(f.fold_id),
                "train_start": pd.Timestamp(f.train_start),
                "train_end": pd.Timestamp(f.train_end),
                "buffer_start": pd.Timestamp(f.buffer_start),
                "test_start": pd.Timestamp(f.test_start),
                "test_end": pd.Timestamp(f.test_end),
            })
    folds_df = pd.DataFrame(fold_rows).sort_values(["symbol", "fold_id"])
    folds_df.to_parquet(tf_snap / "folds_plan.parquet", index=False)

    # 3) Save registry snapshot
    reg_rows = []
    for s in ctx.registry:
        reg_rows.append({
            "strategy_id": s.strategy_id,
            "code": s.code,
            "name": s.name,
            "group": s.group,
            "class_name": s.cls.__name__,
            "save_heatmap": bool(s.save_heatmap),
            "param_grid_json": json.dumps(s.param_grid, ensure_ascii=False, sort_keys=True),
            "has_constraint": bool(s.constraint is not None),
        })
    pd.DataFrame(reg_rows).to_parquet(tf_snap / "registry_snapshot.parquet", index=False)

    # 4) Save data snapshot
    data_rows = []
    for sym in ctx.symbols:
        csv_path = Path(ctx.cfg.data_paths[sym])
        df = ctx.data_by_symbol[sym]
        data_rows.append({
            "symbol": sym,
            "csv_path": str(csv_path),
            "csv_exists": csv_path.exists(),
            "csv_sha256": sha256_file(csv_path) if csv_path.exists() else None,
            "n_rows_loaded": int(len(df)),
            "date_min_loaded": str(pd.Timestamp(df.index.min())) if len(df) else None,
            "date_max_loaded": str(pd.Timestamp(df.index.max())) if len(df) else None,
        })
    pd.DataFrame(data_rows).to_parquet(tf_snap / "data_snapshot.parquet", index=False)

    # 5) Optional variable cache (fast restore in same environment)
    if SAVE_PICKLE_CACHE:
        with (tf_snap / "wf_outputs.pkl").open("wb") as f:
            pickle.dump(wf, f, protocol=pickle.HIGHEST_PROTOCOL)

    # 6) Manifest
    manifest = {
        "created_at_utc": str(pd.Timestamp.utcnow()),
        "timeframe": tf,
        "project_root": str(PROJECT_ROOT),
        "git_head": git_head_or_none(),
        "python": platform.python_version(),
        "symbols": ctx.symbols,
        "costs": ctx.cfg.costs,
        "periods_per_year": int(ctx.ppy),
        "wf": asdict(ctx.cfg.wf),
        "objective": asdict(ctx.cfg.objective),
        "rc_spa": asdict(ctx.cfg.rc_spa),
        "artifacts_copied": copied,
        "pickle_saved": bool(SAVE_PICKLE_CACHE),
        "shapes": {
            "folds_best": tuple(wf.folds_best.shape),
            "returns_oos": tuple(wf.returns_oos.shape),
            "bench_oos": tuple(wf.bench_oos.shape),
            "bench_folds": tuple(wf.bench_folds.shape),
            "heatmaps": None if wf.heatmaps is None else tuple(wf.heatmaps.shape),
        },
    }
    (tf_snap / "manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    summary_rows.append({
        "timeframe": tf,
        "snapshot_dir": str(tf_snap),
        "copied_files": len(copied),
        "folds_best_shape": tuple(wf.folds_best.shape),
        "returns_oos_shape": tuple(wf.returns_oos.shape),
        "bench_oos_shape": tuple(wf.bench_oos.shape),
    })

display(pd.DataFrame(summary_rows))
print("PHASE1_SNAPSHOT_ROOT =", PHASE1_SNAPSHOT_ROOT)


C:\Users\prosh\AppData\Local\Temp\ipykernel_25420\839452423.py:32: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  timestamp = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M%S")
C:\Users\prosh\AppData\Local\Temp\ipykernel_25420\839452423.py:114: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "created_at_utc": str(pd.Timestamp.utcnow()),
C:\Users\prosh\AppData\Local\Temp\ipykernel_25420\839452423.py:114: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  "created_at_utc": str(pd.Timestamp.utcnow()),


,timeframe,snapshot_dir,copied_files,folds_best_shape,returns_oos_shape,bench_oos_shape
0,1d,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...,6,"(1656, 23)","(302472, 5)","(25206, 5)"
1,4h,c:\Users\prosh\OneDrive\Рабочий стол\git\diplo...,6,"(1728, 23)","(1894536, 5)","(157878, 5)"


PHASE1_SNAPSHOT_ROOT = c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\runner_exp\_snapshots\phase1_20260402_151053


In [ ]:
# Cell B: Restore contexts + wf_outputs later (without rerunning Phase 1)
import pickle
from pathlib import Path
import pandas as pd

from src.pipeline_exp import build_timeframe_context, WFOutputs

# Option 1: set explicit path from previous cell output
# SNAPSHOT_ROOT = Path(r"...")

# Option 2: auto-pick latest snapshot
snap_base = PROJECT_ROOT / "results" / "runner_exp" / "_snapshots"
snapshots = sorted([p for p in snap_base.glob("phase1_*") if p.is_dir()], key=lambda p: p.stat().st_mtime)
if not snapshots:
    raise FileNotFoundError(f"No snapshots found in {snap_base}")
SNAPSHOT_ROOT = snapshots[-1]

print("Using snapshot:", SNAPSHOT_ROOT)

contexts = {}
wf_outputs = {}
restore_rows = []

for tf in TIMEFRAMES:
    ctx = build_timeframe_context(
        project_root=PROJECT_ROOT,
        timeframe=tf,
        symbol_to_active=SYMBOL_TO_ACTIVE,
        objective_min_trades=OBJECTIVE_MIN_TRADES,
        rc_block_size=RC_BLOCK_SIZE,
        results_root_name="runner_exp",
        logger=logger,
    )
    contexts[tf] = ctx

    tf_snap = SNAPSHOT_ROOT / tf
    pkl_path = tf_snap / "wf_outputs.pkl"

    if pkl_path.exists():
        with pkl_path.open("rb") as f:
            wf = pickle.load(f)
        source = "pickle"
    else:
        src_dir = tf_snap if (tf_snap / "folds_best.parquet").exists() else ctx.cfg.results_dir
        heatmaps_path = src_dir / "train_heatmaps.parquet"
        wf = WFOutputs(
            folds_best=pd.read_parquet(src_dir / "folds_best.parquet"),
            returns_oos=pd.read_parquet(src_dir / "returns_oos.parquet"),
            bench_oos=pd.read_parquet(src_dir / "bench_returns_oos.parquet"),
            bench_folds=pd.read_parquet(src_dir / "bench_folds.parquet"),
            heatmaps=pd.read_parquet(heatmaps_path) if heatmaps_path.exists() else None,
        )
        source = "parquet"

    wf_outputs[tf] = wf
    restore_rows.append({
        "timeframe": tf,
        "source": source,
        "folds_best_shape": tuple(wf.folds_best.shape),
        "returns_oos_shape": tuple(wf.returns_oos.shape),
        "bench_oos_shape": tuple(wf.bench_oos.shape),
        "heatmaps_shape": None if wf.heatmaps is None else tuple(wf.heatmaps.shape),
    })

display(pd.DataFrame(restore_rows))


### Phase 2. Reports, Tables, and Plots


In [8]:
reports_outputs = {}

for tf in TIMEFRAMES:
    logger.info("===== PHASE 2/4 | REPORTS/TABLES/PLOTS | %s =====", tf)
    rep = run_reports_phase(
        contexts[tf],
        wf_outputs[tf],
        top_k_plots=3,
        save_plots=True,
        logger=logger,
    )
    reports_outputs[tf] = rep
    logger.info(
        "REPORTS DONE %s | leaderboard=%s | stitched=%s",
        tf,
        rep.leaderboard.shape,
        rep.stitched.shape,
    )

list(reports_outputs.keys())


2026-04-02 18:14:03,436 | INFO | ===== PHASE 2/4 | REPORTS/TABLES/PLOTS | 1d =====
2026-04-02 18:14:23,631 | INFO | REPORTS DONE 1d | leaderboard=(180, 36) | stitched=(180, 12)
2026-04-02 18:14:23,633 | INFO | REPORTS DONE 1d | leaderboard=(180, 36) | stitched=(180, 12)
2026-04-02 18:14:23,634 | INFO | ===== PHASE 2/4 | REPORTS/TABLES/PLOTS | 4h =====
2026-04-02 18:14:50,266 | INFO | REPORTS DONE 4h | leaderboard=(180, 36) | stitched=(180, 12)
2026-04-02 18:14:50,267 | INFO | REPORTS DONE 4h | leaderboard=(180, 36) | stitched=(180, 12)


['1d', '4h']

### Phase 3. RC/SPA Tests


In [9]:
tests_outputs = {}

for tf in TIMEFRAMES:
    logger.info("===== PHASE 3/4 | RC/SPA TESTS | %s =====", tf)
    tst = run_tests_phase(
        contexts[tf],
        wf_outputs[tf],
        lambdas=LAMBDA_LIST,
        lam_star=LAM_STAR,
        alpha=ALPHA,
        pvalue_type=PV_TYPE,
        max_show=MAX_SHOW,
        logger=logger,
    )
    tests_outputs[tf] = tst
    logger.info(
        "TESTS DONE %s | rcspa=%s | rcspa_util=%s",
        tf,
        tst.rcspa.shape,
        tst.rcspa_util.shape,
    )

list(tests_outputs.keys())


2026-04-02 18:18:46,788 | INFO | ===== PHASE 3/4 | RC/SPA TESTS | 1d =====
2026-04-02 18:18:46,875 | INFO | RC/SPA universe: BTCUSDT | Low | T=1826 | models=12
2026-04-02 18:18:59,506 | INFO | RC/SPA universe: BTCUSDT | Base | T=1826 | models=12
2026-04-02 18:19:04,839 | INFO | RC/SPA universe: BTCUSDT | High | T=1826 | models=12
2026-04-02 18:19:10,227 | INFO | RC/SPA universe: ETHUSDT | Low | T=1826 | models=12
2026-04-02 18:19:15,630 | INFO | RC/SPA universe: ETHUSDT | Base | T=1826 | models=12
2026-04-02 18:19:20,876 | INFO | RC/SPA universe: ETHUSDT | High | T=1826 | models=12
2026-04-02 18:19:26,221 | INFO | RC/SPA universe: BNBUSDT | Low | T=1826 | models=12
2026-04-02 18:19:31,363 | INFO | RC/SPA universe: BNBUSDT | Base | T=1826 | models=12
2026-04-02 18:19:36,823 | INFO | RC/SPA universe: BNBUSDT | High | T=1826 | models=12
2026-04-02 18:19:42,067 | INFO | RC/SPA universe: XRPUSDT | Low | T=1644 | models=12
2026-04-02 18:19:46,760 | INFO | RC/SPA universe: XRPUSDT | Base | T=

['1d', '4h']

### Phase 4. Robustness


In [10]:
robustness_outputs = {}

for tf in TIMEFRAMES:
    logger.info("===== PHASE 4/4 | ROBUSTNESS | %s =====", tf)
    rob = run_robustness_phase(
        contexts[tf],
        wf_outputs[tf],
        reports_outputs[tf],
        robust_min_bars=ROBUST_MIN_BARS[tf],
        subperiods=SUBPERIODS,
        lam_star=LAM_STAR,
        top_n=5,
        top_k=3,
        logger=logger,
    )
    robustness_outputs[tf] = rob
    logger.info(
        "ROBUST DONE %s | sens_cal=%s | sub_cal=%s",
        tf,
        rob.sensitivity_calmar.shape,
        rob.subperiod_calmar.shape,
    )

list(robustness_outputs.keys())


2026-04-02 19:55:48,619 | INFO | ===== PHASE 4/4 | ROBUSTNESS | 1d =====
2026-04-02 19:55:55,389 | INFO | ROBUST DONE 1d | sens_cal=(90, 13) | sub_cal=(48, 13)
2026-04-02 19:55:55,390 | INFO | ROBUST DONE 1d | sens_cal=(90, 13) | sub_cal=(48, 13)
2026-04-02 19:55:55,391 | INFO | ===== PHASE 4/4 | ROBUSTNESS | 4h =====
2026-04-02 19:56:19,600 | INFO | ROBUST DONE 4h | sens_cal=(90, 13) | sub_cal=(48, 13)
2026-04-02 19:56:19,601 | INFO | ROBUST DONE 4h | sens_cal=(90, 13) | sub_cal=(48, 13)


['1d', '4h']

## Build Combined Summaries


In [11]:
combined_dir = PROJECT_ROOT / "results" / "runner_exp" / "combined"
combined_dir.mkdir(parents=True, exist_ok=True)


def concat_with_tf(outputs_by_tf: dict, attr: str) -> pd.DataFrame:
    parts = []
    for tf, out in outputs_by_tf.items():
        df = getattr(out, attr, pd.DataFrame())
        if df is None or len(df) == 0:
            continue
        x = df.copy()
        x["timeframe"] = tf
        parts.append(x)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


combined = {
    "leaderboard": concat_with_tf(reports_outputs, "leaderboard"),
    "stitched": concat_with_tf(reports_outputs, "stitched"),
    "bench_stitched": concat_with_tf(reports_outputs, "bench_stitched"),
    "rcspa": concat_with_tf(tests_outputs, "rcspa"),
    "rcspa_util": concat_with_tf(tests_outputs, "rcspa_util"),
    "spa_better_table": concat_with_tf(tests_outputs, "spa_tbl"),
    "robust_sensitivity_calmar": concat_with_tf(robustness_outputs, "sensitivity_calmar"),
    "robust_sensitivity_utility": concat_with_tf(robustness_outputs, "sensitivity_utility"),
    "robust_subperiod_calmar": concat_with_tf(robustness_outputs, "subperiod_calmar"),
    "robust_subperiod_utility": concat_with_tf(robustness_outputs, "subperiod_utility"),
}

for name, df in combined.items():
    safe_write_parquet(df, combined_dir / f"{name}.parquet")

shape_rows = [{"table": k, "shape": tuple(v.shape)} for k, v in combined.items()]
shape_df = pd.DataFrame(shape_rows)
display(shape_df)

display(combined["stitched"].head(20))


,table,shape
0,leaderboard,"(360, 37)"
1,stitched,"(360, 13)"
2,bench_stitched,"(30, 13)"
3,rcspa,"(30, 15)"
4,rcspa_util,"(90, 16)"
5,spa_better_table,"(355, 15)"
6,robust_sensitivity_calmar,"(180, 14)"
7,robust_sensitivity_utility,"(180, 15)"
8,robust_subperiod_calmar,"(96, 14)"
9,robust_subperiod_utility,"(96, 15)"


,symbol,cost,strategy_id,n_bars,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,MaxDD,Calmar,CDaR_95,timeframe
0,BNBUSDT,Base,T3:Donchian_Breakout,1826,35.417036,1.051587,0.675672,1.371293,2.852634,-0.613364,1.714457,0.576029,1d
1,BNBUSDT,Base,T2:EMA_Crossover,1826,32.740751,1.020522,0.735202,1.299395,2.512035,-0.614405,1.660992,0.547283,1d
2,BNBUSDT,Base,M1:TSMOM,1826,23.405776,0.893854,0.697467,1.235261,2.514159,-0.560726,1.594101,0.534688,1d
3,BNBUSDT,Base,T1:SMA_Crossover,1826,27.341749,0.951310,0.717001,1.267592,2.414197,-0.607747,1.565306,0.596614,1d
4,BNBUSDT,Base,S3:Breakout_Confirm_MA,1826,16.822676,0.778515,0.678180,1.163134,2.243679,-0.635869,1.224332,0.625218,1d
5,BNBUSDT,Base,S4:MA_Confirm_TSMOM,1826,11.076940,0.645405,0.733124,1.020734,1.932753,-0.759985,0.849234,0.742353,1d
6,BNBUSDT,Base,R2:Bollinger_MR,1826,2.775377,0.304154,0.398783,0.862514,1.362694,-0.385788,0.788396,0.286145,1d
7,BNBUSDT,Base,S5:Ensemble_3Signals,1826,5.575616,0.457128,0.752640,0.853021,1.564932,-0.850652,0.537385,0.847665,1d
8,BNBUSDT,Base,S1:MAFilter_RSI_MR,1826,0.850779,0.130945,0.283973,0.568757,1.011656,-0.333487,0.392656,0.063499,1d
9,BNBUSDT,Base,S2:MA200Filter_Bollinger_MR,1826,0.441299,0.075805,0.190680,0.477751,0.748556,-0.227916,0.332600,0.114762,1d


In [12]:
# Quick sanity: 4h annualization differs from 365
r = pd.Series([0.001] * 200)
m_365 = compute_metrics_from_returns(r, periods_per_year=365)
m_2190 = compute_metrics_from_returns(r, periods_per_year=2190)
pd.DataFrame({
    "metric": ["CAGR", "Ann. Vol", "Sharpe", "Sortino", "Calmar"],
    "ppy_365": [m_365["CAGR"], m_365["Ann. Vol"], m_365["Sharpe"], m_365["Sortino"], m_365["Calmar"]],
    "ppy_2190": [m_2190["CAGR"], m_2190["Ann. Vol"], m_2190["Sharpe"], m_2190["Sortino"], m_2190["Calmar"]],
})


,metric,ppy_365,ppy_2190
0,CAGR,4.402513e-01,7.925441e+00
1,Ann. Vol,4.142731e-18,1.014758e-17
2,Sharpe,8.810614e+16,2.158151e+17
3,Sortino,0.000000e+00,0.000000e+00
4,Calmar,0.000000e+00,0.000000e+00
